In [1]:
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import wandb
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
class CustomTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"

        self.word2idx ={self.pad_token: 0, self.unk_token: 1}
        self.idx2word = {0: self.pad_token, 1: self.unk_token}

    def build_vocab(self, sentences):
        token_counts = Counter(token for sentence in sentences for token in sentence)

        for token in token_counts:
            if token not in self.word2idx: # 특수 토큰 및 이미 추가된 토큰 제외
                self.word2idx[token] = len(self.word2idx)
                self.idx2word[len(self.idx2word)] = token

        print(f"Vocab size: {self.vocab_size}")
        print(f"Example tokens: {list(self.word2idx.keys())[:10]}")

    def encode(self, tokens):
        return [self.word2idx.get(token, self.word2idx[self.unk_token]) for token in tokens]

    def decode(self, ids):
        return [self.idx2word.get(idx, self.word2idx[self.unk_token]) for idx in ids]

    @property
    def vocab_size(self):
        return len(self.word2idx)

    @property
    def pad_token_id(self):
        return self.word2idx[self.pad_token]

    @property
    def unk_token_id(self):
        return self.word2idx[self.unk_token]

In [3]:
class NERDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_len=64):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.ner_feature = self.dataset.features['ner_tags']
        self.id2tag = {id: tag for id, tag in enumerate(self.ner_feature.feature.names)}
        self.tag2id = {tag: id for id, tag in self.id2tag.items()}
        self.num_labels = self.ner_feature.feature.num_classes
        self.ignore_index = -100

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        
        tokens = example['tokens']
        labels = example['ner_tags']

        input_ids = self.tokenizer.encode(tokens)

        input_ids = input_ids[:self.max_len]
        labels = labels[:self.max_len]

        input_ids += [self.tokenizer.pad_token_id] * (self.max_len - len(input_ids))
        labels += [self.ignore_index] * (self.max_len - len(labels))

        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = torch.tensor(labels, dtype=torch.long)

        return input_ids, labels

In [4]:
class TextCNN_NER(nn.Module):
    def __init__(self, vocab_size, embedding_lookup_matrix, num_labels, activation, hidden_dim=32, embedding_dim=32):
        super(TextCNN_NER, self).__init__()

        self.num_labels = num_labels

        self.SG_embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embedding_lookup_matrix), freeze=True)
        self.RD_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.SG_conv1 = nn.Conv2d(1, hidden_dim, (3, hidden_dim), padding=(1, 0)) # hidden_dim -> embedding_dim
        self.SG_conv2 = nn.Conv2d(1, hidden_dim, (5, hidden_dim), padding=(2, 0))
        self.SG_conv3 = nn.Conv2d(1, hidden_dim, (7, hidden_dim), padding=(3, 0))

        self.RD_conv1 = nn.Conv2d(1, hidden_dim, (3, hidden_dim), padding=(1, 0))
        self.RD_conv2 = nn.Conv2d(1, hidden_dim, (5, hidden_dim), padding=(2, 0))
        self.RD_conv3 = nn.Conv2d(1, hidden_dim, (7, hidden_dim), padding=(3, 0))
        
        self.fc = nn.Linear(6 * hidden_dim, self.num_labels)
        self.activation = activation

    def forward(self, input_ids):
        SG_embedding = self.SG_embedding(input_ids).unsqueeze(1)
        RD_embedding = self.RD_embedding(input_ids).unsqueeze(1)

        SG_conv1_feature = self.activation(self.SG_conv1(SG_embedding)).squeeze(3)
        SG_conv2_feature = self.activation(self.SG_conv2(SG_embedding)).squeeze(3)
        SG_conv3_feature = self.activation(self.SG_conv3(SG_embedding)).squeeze(3)

        RD_conv1_feature = self.activation(self.RD_conv1(RD_embedding)).squeeze(3)
        RD_conv2_feature = self.activation(self.RD_conv2(RD_embedding)).squeeze(3)
        RD_conv3_feature = self.activation(self.RD_conv3(RD_embedding)).squeeze(3)

        x = torch.cat([SG_conv1_feature, SG_conv2_feature, SG_conv3_feature,
                       RD_conv1_feature, RD_conv2_feature, RD_conv3_feature], dim=1)
        x = x.permute(0, 2, 1)
        x = self.fc(x)

        return x

In [5]:
def evaluate(model, dataloader, device, id2tag):
    model.eval()
    correct_predictions = 0
    total_predictions = 0
    all_predicted_labels = []
    all_true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids, labels = batch
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            outputs = model(input_ids)
            outputs = outputs.view(-1, 9)
            labels = labels.view(-1)

            _, predicted_labels = torch.max(outputs, 1)

            mask = labels != -100
            
            predicted_labels = predicted_labels[mask].cpu().numpy()
            true_labels = labels[mask].cpu().numpy()
            
            correct_predictions += (predicted_labels == true_labels).sum().item()
            total_predictions += len(true_labels)
            
            all_predicted_labels.extend(predicted_labels)
            all_true_labels.extend(true_labels)

    accuracy = correct_predictions / total_predictions

    micro_f1 = f1_score(all_true_labels, all_predicted_labels, average='micro')
    macro_f1 = f1_score(all_true_labels, all_predicted_labels, average='macro')

    print(f"## Evaluation Results ##")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Micro F1: {micro_f1:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")


    wandb.log({"val_accuracy": accuracy})
    wandb.log({"val_micro_f1": micro_f1})
    wandb.log({"val_macro_f1": macro_f1})
    
    print("## Detailed Performance by Entity Type ##")
    class_results = {}
    for i in range(9):
        class_name = id2tag[i]
        class_precision = precision_score(all_true_labels, all_predicted_labels, labels=[i], average='macro')
        class_recall = recall_score(all_true_labels, all_predicted_labels, labels=[i], average='macro')
        class_f1 = f1_score(all_true_labels, all_predicted_labels, labels=[i], average='macro')
        class_support = sum(1 for label in all_true_labels if label == i)
        class_results[class_name] = {
            "precision": class_precision,
            "recall": class_recall,
            "f1": class_f1,
            "support": class_support
        }
        print(f"{class_name}: Precision={class_precision:.4f}, Recall={class_recall:.4f}, F1={class_f1:.4f}, Support={class_support}")
    print()
    return accuracy, micro_f1, macro_f1, class_results

In [6]:
def train(model, train_dataloader, valid_dataloader, test_dataloader, optimizer, criterion, device):
    print("Evaluate before training...")
    results = evaluate(model, test_dataloader, device, train_dataloader.dataset.id2tag)
    print("Evaluation finised.\n")

    print("Start training...")
    for epoch in range(wandb.config.num_epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_dataloader):
            input_ids, labels = batch
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(input_ids)
            outputs = outputs.view(-1, 9)
            labels = labels.view(-1)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            wandb.log({"train_loss_step": loss.item()})
            
        if (epoch + 1) % wandb.config.validation_epoch == 0:
            evaluate(model, valid_dataloader, device, train_dataloader.dataset.id2tag)

        print(f"Epoch {epoch + 1}/{wandb.config.num_epochs}, Loss: {total_loss / len(train_dataloader)}")
        wandb.log({"train_loss_epoch": total_loss / len(train_dataloader)})
        wandb.log({"epoch": epoch + 1})
    print("Training finished.\n")

    print("Evaluate after training...")
    results = evaluate(model, test_dataloader, device, train_dataloader.dataset.id2tag)
    print("Evaluation finished.\n")
    print("Training and evaluation completed.")

    return model

In [7]:
class Optimizer:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.detach_()
                param.grad.zero_()

    def step(self):
        raise NotImplementedError("Optimizer step method not implemented.")


class Momentum(Optimizer):
    def __init__(self, params, lr, momentum=0.9):
        super().__init__(params, lr)
        self.momentum = momentum
        self.velocities = [torch.zeros_like(param) for param in self.params]

    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                grad = param.grad
                self.velocities[i] = self.momentum * self.velocities[i] + grad
                param.add_(self.lr * self.velocities[i])


class NesterovMomentum(Optimizer):
    def __init__(self, params, lr, momentum=0.9):
        super().__init__(params, lr)
        self.momentum = momentum
        self.velocity = {param: torch.zeros_like(param) for param in self.params}
    
    # Served
#     def step(self):
#         with torch.no_grad():
#             for i, param in enumerate(self.params):
#                 if param.grad is None:
#                     continue
#                 grad = param.grad
#                 lookahead_param = param + self.momentum * self.velocities[i]
                
#                 param_data = param.data.clone()
#                 param_data = lookahead_param
                
#                 lookahead_grad = param.grad.clone()
                
#                 param_data = param_data
#                 param_grad = grad
                
#                 prev_velocity = self.velocities[i].clone()
#                 self.velocities[i] = self.momentum * self.velocities[i] - self.lr * lookahead_grad
                
#                 param.add_(self.velocities[i] + self.momentum * (self.velocities[i] - self.prev_velocity))
                
    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                prev_velocity = self.velocities[i].clone()
                self.velocities[i] = self.momentum * self.velocities[i] - self.lr * param.grad
                param.add_(-self.momentum * prev_velocity + (1 + self.momentum) * self.velocities[i])


class AdaGrad(Optimizer):
    def __init__(self, params, lr, eps=1e-10):
        super().__init__(params, lr)
        self.eps = eps
        self.h = [torch.zeros_like(param) for param in self.params]

    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.h[i] += param.grad ** 2
                param.add_(-self.lr * param.grad / (torch.sqrt(self.h[i]) + self.eps))


class RMSProp(Optimizer):
    def __init__(self, params, lr, alpha=0.99, eps=1e-8):
        super().__init__(params, lr)
        self.alpha = alpha
        self.eps = eps
        self.v = [torch.zeros_like(param) for param in self.params]

    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.v[i] = self.alpha * self.v[i] + (1 - self.alpha) * param.grad ** 2
                param.add_(-self.lr * param.grad / (torch.sqrt(self.v[i]) + self.eps))


class Adam(Optimizer):
    def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = [torch.zeros_like(param) for param in self.params]
        self.v = [torch.zeros_like(param) for param in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * param.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * param.grad ** 2

                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                param.add_(-self.lr * m_hat / (torch.sqrt(v_hat) + self.eps))


class AdamW(Optimizer):
    def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=1e-2):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = [torch.zeros_like(param) for param in self.params]
        self.v = [torch.zeros_like(param) for param in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * param.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * param.grad ** 2

                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                param.add_(-self.lr * (m_hat / (torch.sqrt(v_hat) + self.eps) + self.weight_decay * param)) 

In [8]:
def relu(x):
    return torch.max(torch.zeros_like(x), x)

def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def tanh(x):
    return torch.tanh(x)

In [15]:
config = {
    'batch_size': 16, # Served : 32
    'learning_rate': 5e-3, # Served : 1e-2
    'num_epochs': 20,
    'max_len': 128, # Served : 64
    'validation_epoch': 2, # Served : 2
    'activation': "ReLU",    # sigmoid, tanh, ReLU
    'optimizer': "AdamW",    # Momentum, NesterovMomentum, AdaGrad, RMSProp, Adam, AdamW
    'embedding_dim': 32, # Served : 64
    'hidden_dim': 32, # Served : 64
}

wandb.init(
    project="CSE541",
    name="NER-TextCNN",
    group="NER",
    tags=["NER", "TextCNN"],
    config=config,
    mode="disabled"    
)

In [16]:
data = load_dataset("eriktks/conll2003")
train_sentences = [example['tokens'] for example in data['train']]

tokenizer = CustomTokenizer()
tokenizer.build_vocab(train_sentences)

from gensim.models import Word2Vec

# CBOW_W2V = Word2Vec(sentences=data['train'][:]['tokens'], vector_size=wandb.config.embedding_dim, window=5, min_count=1, workers=4, sg=0) # CBOW
SkipGram_W2V = Word2Vec(sentences=data['train'][:]['tokens'], vector_size=wandb.config.embedding_dim, window=5, min_count=1, workers=4, sg=1) # SkipGram

embedding_lookup_matrix = []

for token, idx in tokenizer.word2idx.items():
    if token in SkipGram_W2V.wv:
        embedding_lookup_matrix.append(SkipGram_W2V.wv[token])
    elif token == "[PAD]":
        embedding_lookup_matrix.append(np.zeros(SkipGram_W2V.wv.vectors.shape[1]))
    elif token == "[UNK]":
        embedding_lookup_matrix.append(np.random.uniform(-1, 1, SkipGram_W2V.wv.vectors.shape[1]))
    else:
        embedding_lookup_matrix.append(np.random.uniform(-1, 1, SkipGram_W2V.wv.vectors.shape[1]))

embedding_lookup_matrix = np.vstack(embedding_lookup_matrix)

train_dataset = NERDataset(data['train'], tokenizer, max_len=wandb.config.max_len)
valid_dataset = NERDataset(data['validation'], tokenizer, max_len=wandb.config.max_len)
test_dataset = NERDataset(data['test'], tokenizer, max_len=wandb.config.max_len)

train_dataloader = DataLoader(train_dataset, batch_size=wandb.config.batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=wandb.config.batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=wandb.config.batch_size, shuffle=False)

Vocab size: 23625
Example tokens: ['<PAD>', '<UNK>', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb']


In [17]:
if wandb.config.activation == "ReLU":
    activation = relu
elif wandb.config.activation == "sigmoid":
    activation = sigmoid
elif wandb.config.activation == "tanh":
    activation = tanh

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextCNN_NER(len(tokenizer.word2idx), embedding_lookup_matrix, train_dataset.num_labels, activation, wandb.config.hidden_dim, wandb.config.embedding_dim).to(device)

model_params = model.parameters()

if wandb.config.optimizer == "Momentum":
    optimizer = Momentum(model_params, lr=wandb.config.learning_rate, momentum=0.9)
elif wandb.config.optimizer == "NesterovMomentum":
    optimizer = NesterovMomentum(model_params, lr=wandb.config.learning_rate, momentum=0.9)
elif wandb.config.optimizer == "AdaGrad":
    optimizer = AdaGrad(model_params, lr=wandb.config.learning_rate)
elif wandb.config.optimizer == "RMSProp":
    optimizer = RMSProp(model_params, lr=wandb.config.learning_rate, alpha=0.99, eps=1e-8)
elif wandb.config.optimizer == "Adam":
    optimizer = Adam(model_params, lr=wandb.config.learning_rate, beta1=0.9, beta2=0.999, eps=1e-8)
elif wandb.config.optimizer == "AdamW":
    optimizer = AdamW(model_params, lr=wandb.config.learning_rate, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

In [18]:
model = train(model, train_dataloader, valid_dataloader, test_dataloader, optimizer, criterion, device)

Evaluate before training...


100% 216/216 [00:00<00:00, 290.63it/s]


## Evaluation Results ##
Accuracy: 0.0386
Micro F1: 0.0386
Macro F1: 0.0183
## Detailed Performance by Entity Type ##
O: Precision=0.8672, Recall=0.0372, F1=0.0713, Support=38323
B-PER: Precision=0.0250, Recall=0.0006, F1=0.0012, Support=1617
I-PER: Precision=0.0126, Recall=0.0164, F1=0.0143, Support=1156
B-ORG: Precision=0.0000, Recall=0.0000, F1=0.0000, Support=1661
I-ORG: Precision=0.0079, Recall=0.0012, F1=0.0021, Support=835
B-LOC: Precision=0.0305, Recall=0.0869, F1=0.0451, Support=1668
I-LOC: Precision=0.0062, Recall=0.4786, F1=0.0123, Support=257
B-MISC: Precision=0.0066, Recall=0.0271, F1=0.0106, Support=702
I-MISC: Precision=0.0039, Recall=0.2824, F1=0.0076, Support=216

Evaluation finised.

Start training...


100% 878/878 [00:09<00:00, 88.92it/s]


Epoch 1/20, Loss: 0.33400591958424497


...

Epoch 19/20, Loss: 0.002241156881750066


100% 878/878 [00:09<00:00, 89.95it/s]
100% 204/204 [00:00<00:00, 275.30it/s]


## Evaluation Results ##
Accuracy: 0.9579
Micro F1: 0.9579
Macro F1: 0.8287
## Detailed Performance by Entity Type ##
O: Precision=0.9727, Recall=0.9944, F1=0.9834, Support=42759
B-PER: Precision=0.9328, Recall=0.7682, F1=0.8425, Support=1842
I-PER: Precision=0.9591, Recall=0.7536, F1=0.8440, Support=1307
B-ORG: Precision=0.6823, Recall=0.8024, F1=0.7375, Support=1341
I-ORG: Precision=0.9204, Recall=0.6312, F1=0.7488, Support=751
B-LOC: Precision=0.9205, Recall=0.8568, F1=0.8875, Support=1837
I-LOC: Precision=0.8959, Recall=0.7704, F1=0.8285, Support=257
B-MISC: Precision=0.8901, Recall=0.7993, F1=0.8423, Support=922
I-MISC: Precision=0.8780, Recall=0.6445, F1=0.7433, Support=346

Epoch 20/20, Loss: 0.002651331760518666
Training finished.

Evaluate after training...


100% 216/216 [00:00<00:00, 291.85it/s]


## Evaluation Results ##
Accuracy: 0.9341
Micro F1: 0.9341
Macro F1: 0.7473
## Detailed Performance by Entity Type ##
O: Precision=0.9607, Recall=0.9875, F1=0.9739, Support=38323
B-PER: Precision=0.8530, Recall=0.6314, F1=0.7257, Support=1617
I-PER: Precision=0.9179, Recall=0.5900, F1=0.7183, Support=1156
B-ORG: Precision=0.6038, Recall=0.7092, F1=0.6523, Support=1661
I-ORG: Precision=0.8564, Recall=0.6144, F1=0.7155, Support=835
B-LOC: Precision=0.8690, Recall=0.8076, F1=0.8372, Support=1668
I-LOC: Precision=0.8418, Recall=0.6420, F1=0.7285, Support=257
B-MISC: Precision=0.7919, Recall=0.7208, F1=0.7547, Support=702
I-MISC: Precision=0.7083, Recall=0.5509, F1=0.6198, Support=216

Evaluation finished.

Training and evaluation completed.


In [19]:
results = evaluate(model, test_dataloader, device, train_dataloader.dataset.id2tag)

100% 216/216 [00:00<00:00, 290.75it/s]


## Evaluation Results ##
Accuracy: 0.9341
Micro F1: 0.9341
Macro F1: 0.7473
## Detailed Performance by Entity Type ##
O: Precision=0.9607, Recall=0.9875, F1=0.9739, Support=38323
B-PER: Precision=0.8530, Recall=0.6314, F1=0.7257, Support=1617
I-PER: Precision=0.9179, Recall=0.5900, F1=0.7183, Support=1156
B-ORG: Precision=0.6038, Recall=0.7092, F1=0.6523, Support=1661
I-ORG: Precision=0.8564, Recall=0.6144, F1=0.7155, Support=835
B-LOC: Precision=0.8690, Recall=0.8076, F1=0.8372, Support=1668
I-LOC: Precision=0.8418, Recall=0.6420, F1=0.7285, Support=257
B-MISC: Precision=0.7919, Recall=0.7208, F1=0.7547, Support=702
I-MISC: Precision=0.7083, Recall=0.5509, F1=0.6198, Support=216



In [14]:
wandb.finish()

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import wandb
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
class CustomTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"

        self.word2idx ={self.pad_token: 0, self.unk_token: 1}
        self.idx2word = {0: self.pad_token, 1: self.unk_token}

    def build_vocab(self, sentences):
        token_counts = Counter(token for sentence in sentences for token in sentence)

        for token in token_counts:
            if token not in self.word2idx: # 특수 토큰 및 이미 추가된 토큰 제외
                self.word2idx[token] = len(self.word2idx)
                self.idx2word[len(self.idx2word)] = token

        print(f"Vocab size: {self.vocab_size}")
        print(f"Example tokens: {list(self.word2idx.keys())[:10]}")

    def encode(self, tokens):
        return [self.word2idx.get(token, self.word2idx[self.unk_token]) for token in tokens]

    def decode(self, ids):
        return [self.idx2word.get(idx, self.word2idx[self.unk_token]) for idx in ids]

    @property
    def vocab_size(self):
        return len(self.word2idx)

    @property
    def pad_token_id(self):
        return self.word2idx[self.pad_token]

    @property
    def unk_token_id(self):
        return self.word2idx[self.unk_token]

In [ ]:
class NERDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_len=64):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.ner_feature = self.dataset.features['ner_tags']
        self.id2tag = {id: tag for id, tag in enumerate(self.ner_feature.feature.names)}
        self.tag2id = {tag: id for id, tag in self.id2tag.items()}
        self.num_labels = self.ner_feature.feature.num_classes
        self.ignore_index = -100

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        
        tokens = example['tokens']
        labels = example['ner_tags']

        input_ids = self.tokenizer.encode(tokens)

        input_ids = input_ids[:self.max_len]
        labels = labels[:self.max_len]

        input_ids += [self.tokenizer.pad_token_id] * (self.max_len - len(input_ids))
        labels += [self.ignore_index] * (self.max_len - len(labels))

        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = torch.tensor(labels, dtype=torch.long)

        return input_ids, labels

In [ ]:
class TextCNN_NER(nn.Module):
    def __init__(self, vocab_size, embedding_lookup_matrix, num_labels, activation, hidden_dim=32, embedding_dim=32):
        super(TextCNN_NER, self).__init__()

        self.num_labels = num_labels

        self.SG_embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embedding_lookup_matrix), freeze=True)
        self.RD_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.SG_conv1 = nn.Conv2d(1, hidden_dim, (3, hidden_dim), padding=(1, 0)) # hidden_dim -> embedding_dim
        self.SG_conv2 = nn.Conv2d(1, hidden_dim, (5, hidden_dim), padding=(2, 0))
        self.SG_conv3 = nn.Conv2d(1, hidden_dim, (7, hidden_dim), padding=(3, 0))

        self.RD_conv1 = nn.Conv2d(1, hidden_dim, (3, hidden_dim), padding=(1, 0))
        self.RD_conv2 = nn.Conv2d(1, hidden_dim, (5, hidden_dim), padding=(2, 0))
        self.RD_conv3 = nn.Conv2d(1, hidden_dim, (7, hidden_dim), padding=(3, 0))
        
        self.fc = nn.Linear(6 * hidden_dim, self.num_labels)
        self.activation = activation

    def forward(self, input_ids):
        SG_embedding = self.SG_embedding(input_ids).unsqueeze(1)
        RD_embedding = self.RD_embedding(input_ids).unsqueeze(1)

        SG_conv1_feature = self.activation(self.SG_conv1(SG_embedding)).squeeze(3)
        SG_conv2_feature = self.activation(self.SG_conv2(SG_embedding)).squeeze(3)
        SG_conv3_feature = self.activation(self.SG_conv3(SG_embedding)).squeeze(3)

        RD_conv1_feature = self.activation(self.RD_conv1(RD_embedding)).squeeze(3)
        RD_conv2_feature = self.activation(self.RD_conv2(RD_embedding)).squeeze(3)
        RD_conv3_feature = self.activation(self.RD_conv3(RD_embedding)).squeeze(3)

        x = torch.cat([SG_conv1_feature, SG_conv2_feature, SG_conv3_feature,
                       RD_conv1_feature, RD_conv2_feature, RD_conv3_feature], dim=1)
        x = x.permute(0, 2, 1)
        x = self.fc(x)

        return x

In [ ]:
def evaluate(model, dataloader, device, id2tag):
    model.eval()
    correct_predictions = 0
    total_predictions = 0
    all_predicted_labels = []
    all_true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids, labels = batch
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            outputs = model(input_ids)
            outputs = outputs.view(-1, 9)
            labels = labels.view(-1)

            _, predicted_labels = torch.max(outputs, 1)

            mask = labels != -100
            
            predicted_labels = predicted_labels[mask].cpu().numpy()
            true_labels = labels[mask].cpu().numpy()
            
            correct_predictions += (predicted_labels == true_labels).sum().item()
            total_predictions += len(true_labels)
            
            all_predicted_labels.extend(predicted_labels)
            all_true_labels.extend(true_labels)

    accuracy = correct_predictions / total_predictions

    micro_f1 = f1_score(all_true_labels, all_predicted_labels, average='micro')
    macro_f1 = f1_score(all_true_labels, all_predicted_labels, average='macro')

    print(f"## Evaluation Results ##")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Micro F1: {micro_f1:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")


    wandb.log({"val_accuracy": accuracy})
    wandb.log({"val_micro_f1": micro_f1})
    wandb.log({"val_macro_f1": macro_f1})
    
    print("## Detailed Performance by Entity Type ##")
    class_results = {}
    for i in range(9):
        class_name = id2tag[i]
        class_precision = precision_score(all_true_labels, all_predicted_labels, labels=[i], average='macro')
        class_recall = recall_score(all_true_labels, all_predicted_labels, labels=[i], average='macro')
        class_f1 = f1_score(all_true_labels, all_predicted_labels, labels=[i], average='macro')
        class_support = sum(1 for label in all_true_labels if label == i)
        class_results[class_name] = {
            "precision": class_precision,
            "recall": class_recall,
            "f1": class_f1,
            "support": class_support
        }
        print(f"{class_name}: Precision={class_precision:.4f}, Recall={class_recall:.4f}, F1={class_f1:.4f}, Support={class_support}")
    print()
    return accuracy, micro_f1, macro_f1, class_results

In [ ]:
def train(model, train_dataloader, valid_dataloader, test_dataloader, optimizer, criterion, device):
    print("Evaluate before training...")
    results = evaluate(model, test_dataloader, device, train_dataloader.dataset.id2tag)
    print("Evaluation finised.\n")

    print("Start training...")
    for epoch in range(wandb.config.num_epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_dataloader):
            input_ids, labels = batch
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(input_ids)
            outputs = outputs.view(-1, 9)
            labels = labels.view(-1)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            wandb.log({"train_loss_step": loss.item()})
            
        if (epoch + 1) % wandb.config.validation_epoch == 0:
            evaluate(model, valid_dataloader, device, train_dataloader.dataset.id2tag)

        print(f"Epoch {epoch + 1}/{wandb.config.num_epochs}, Loss: {total_loss / len(train_dataloader)}")
        wandb.log({"train_loss_epoch": total_loss / len(train_dataloader)})
        wandb.log({"epoch": epoch + 1})
    print("Training finished.\n")

    print("Evaluate after training...")
    results = evaluate(model, test_dataloader, device, train_dataloader.dataset.id2tag)
    print("Evaluation finished.\n")
    print("Training and evaluation completed.")

    return model

In [ ]:
class Optimizer:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.detach_()
                param.grad.zero_()

    def step(self):
        raise NotImplementedError("Optimizer step method not implemented.")


class Momentum(Optimizer):
    def __init__(self, params, lr, momentum=0.9):
        super().__init__(params, lr)
        self.momentum = momentum
        self.velocities = [torch.zeros_like(param) for param in self.params]

    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                grad = param.grad
                self.velocities[i] = self.momentum * self.velocities[i] + grad
                param.add_(self.lr * self.velocities[i])


class NesterovMomentum(Optimizer):
    def __init__(self, params, lr, momentum=0.9):
        super().__init__(params, lr)
        self.momentum = momentum
        self.velocity = {param: torch.zeros_like(param) for param in self.params}
    
    # Served
#     def step(self):
#         with torch.no_grad():
#             for i, param in enumerate(self.params):
#                 if param.grad is None:
#                     continue
#                 grad = param.grad
#                 lookahead_param = param + self.momentum * self.velocities[i]
                
#                 param_data = param.data.clone()
#                 param_data = lookahead_param
                
#                 lookahead_grad = param.grad.clone()
                
#                 param_data = param_data
#                 param_grad = grad
                
#                 prev_velocity = self.velocities[i].clone()
#                 self.velocities[i] = self.momentum * self.velocities[i] - self.lr * lookahead_grad
                
#                 param.add_(self.velocities[i] + self.momentum * (self.velocities[i] - self.prev_velocity))
                
    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                prev_velocity = self.velocities[i].clone()
                self.velocities[i] = self.momentum * self.velocities[i] - self.lr * param.grad
                param.add_(-self.momentum * prev_velocity + (1 + self.momentum) * self.velocities[i])


class AdaGrad(Optimizer):
    def __init__(self, params, lr, eps=1e-10):
        super().__init__(params, lr)
        self.eps = eps
        self.h = [torch.zeros_like(param) for param in self.params]

    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.h[i] += param.grad ** 2
                param.add_(-self.lr * param.grad / (torch.sqrt(self.h[i]) + self.eps))


class RMSProp(Optimizer):
    def __init__(self, params, lr, alpha=0.99, eps=1e-8):
        super().__init__(params, lr)
        self.alpha = alpha
        self.eps = eps
        self.v = [torch.zeros_like(param) for param in self.params]

    def step(self):
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.v[i] = self.alpha * self.v[i] + (1 - self.alpha) * param.grad ** 2
                param.add_(-self.lr * param.grad / (torch.sqrt(self.v[i]) + self.eps))


class Adam(Optimizer):
    def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = [torch.zeros_like(param) for param in self.params]
        self.v = [torch.zeros_like(param) for param in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * param.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * param.grad ** 2

                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                param.add_(-self.lr * m_hat / (torch.sqrt(v_hat) + self.eps))


class AdamW(Optimizer):
    def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=1e-2):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = [torch.zeros_like(param) for param in self.params]
        self.v = [torch.zeros_like(param) for param in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * param.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * param.grad ** 2

                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                param.add_(-self.lr * (m_hat / (torch.sqrt(v_hat) + self.eps) + self.weight_decay * param)) 

In [ ]:
def relu(x):
    return torch.max(torch.zeros_like(x), x)

def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def tanh(x):
    return torch.tanh(x)

In [ ]:
config = {
    'batch_size': 16, # Served : 32
    'learning_rate': 5e-3, # Served : 1e-2
    'num_epochs': 20,
    'max_len': 128, # Served : 64
    'validation_epoch': 2, # Served : 2
    'activation': "ReLU",    # sigmoid, tanh, ReLU
    'optimizer': "AdamW",    # Momentum, NesterovMomentum, AdaGrad, RMSProp, Adam, AdamW
    'embedding_dim': 32, # Served : 64
    'hidden_dim': 32, # Served : 64
}

wandb.init(
    project="CSE541",
    name="NER-TextCNN",
    group="NER",
    tags=["NER", "TextCNN"],
    config=config,
    mode="disabled"    
)

In [ ]:
data = load_dataset("eriktks/conll2003")
train_sentences = [example['tokens'] for example in data['train']]

tokenizer = CustomTokenizer()
tokenizer.build_vocab(train_sentences)

from gensim.models import Word2Vec

# CBOW_W2V = Word2Vec(sentences=data['train'][:]['tokens'], vector_size=wandb.config.embedding_dim, window=5, min_count=1, workers=4, sg=0) # CBOW
SkipGram_W2V = Word2Vec(sentences=data['train'][:]['tokens'], vector_size=wandb.config.embedding_dim, window=5, min_count=1, workers=4, sg=1) # SkipGram

embedding_lookup_matrix = []

for token, idx in tokenizer.word2idx.items():
    if token in SkipGram_W2V.wv:
        embedding_lookup_matrix.append(SkipGram_W2V.wv[token])
    elif token == "[PAD]":
        embedding_lookup_matrix.append(np.zeros(SkipGram_W2V.wv.vectors.shape[1]))
    elif token == "[UNK]":
        embedding_lookup_matrix.append(np.random.uniform(-1, 1, SkipGram_W2V.wv.vectors.shape[1]))
    else:
        embedding_lookup_matrix.append(np.random.uniform(-1, 1, SkipGram_W2V.wv.vectors.shape[1]))

embedding_lookup_matrix = np.vstack(embedding_lookup_matrix)

train_dataset = NERDataset(data['train'], tokenizer, max_len=wandb.config.max_len)
valid_dataset = NERDataset(data['validation'], tokenizer, max_len=wandb.config.max_len)
test_dataset = NERDataset(data['test'], tokenizer, max_len=wandb.config.max_len)

train_dataloader = DataLoader(train_dataset, batch_size=wandb.config.batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=wandb.config.batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=wandb.config.batch_size, shuffle=False)

Vocab size: 23625
Example tokens: ['<PAD>', '<UNK>', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb']


In [ ]:
if wandb.config.activation == "ReLU":
    activation = relu
elif wandb.config.activation == "sigmoid":
    activation = sigmoid
elif wandb.config.activation == "tanh":
    activation = tanh

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextCNN_NER(len(tokenizer.word2idx), embedding_lookup_matrix, train_dataset.num_labels, activation, wandb.config.hidden_dim, wandb.config.embedding_dim).to(device)

model_params = model.parameters()

if wandb.config.optimizer == "Momentum":
    optimizer = Momentum(model_params, lr=wandb.config.learning_rate, momentum=0.9)
elif wandb.config.optimizer == "NesterovMomentum":
    optimizer = NesterovMomentum(model_params, lr=wandb.config.learning_rate, momentum=0.9)
elif wandb.config.optimizer == "AdaGrad":
    optimizer = AdaGrad(model_params, lr=wandb.config.learning_rate)
elif wandb.config.optimizer == "RMSProp":
    optimizer = RMSProp(model_params, lr=wandb.config.learning_rate, alpha=0.99, eps=1e-8)
elif wandb.config.optimizer == "Adam":
    optimizer = Adam(model_params, lr=wandb.config.learning_rate, beta1=0.9, beta2=0.999, eps=1e-8)
elif wandb.config.optimizer == "AdamW":
    optimizer = AdamW(model_params, lr=wandb.config.learning_rate, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

In [ ]:
model = train(model, train_dataloader, valid_dataloader, test_dataloader, optimizer, criterion, device)

Evaluate before training...


100% 216/216 [00:00<00:00, 290.63it/s]


## Evaluation Results ##
Accuracy: 0.0386
Micro F1: 0.0386
Macro F1: 0.0183
## Detailed Performance by Entity Type ##
O: Precision=0.8672, Recall=0.0372, F1=0.0713, Support=38323
B-PER: Precision=0.0250, Recall=0.0006, F1=0.0012, Support=1617
I-PER: Precision=0.0126, Recall=0.0164, F1=0.0143, Support=1156
B-ORG: Precision=0.0000, Recall=0.0000, F1=0.0000, Support=1661
I-ORG: Precision=0.0079, Recall=0.0012, F1=0.0021, Support=835
B-LOC: Precision=0.0305, Recall=0.0869, F1=0.0451, Support=1668
I-LOC: Precision=0.0062, Recall=0.4786, F1=0.0123, Support=257
B-MISC: Precision=0.0066, Recall=0.0271, F1=0.0106, Support=702
I-MISC: Precision=0.0039, Recall=0.2824, F1=0.0076, Support=216

Evaluation finised.

Start training...


100% 878/878 [00:09<00:00, 88.92it/s]


Epoch 1/20, Loss: 0.33400591958424497


...

Epoch 19/20, Loss: 0.002241156881750066


100% 878/878 [00:09<00:00, 89.95it/s]
100% 204/204 [00:00<00:00, 275.30it/s]


## Evaluation Results ##
Accuracy: 0.9579
Micro F1: 0.9579
Macro F1: 0.8287
## Detailed Performance by Entity Type ##
O: Precision=0.9727, Recall=0.9944, F1=0.9834, Support=42759
B-PER: Precision=0.9328, Recall=0.7682, F1=0.8425, Support=1842
I-PER: Precision=0.9591, Recall=0.7536, F1=0.8440, Support=1307
B-ORG: Precision=0.6823, Recall=0.8024, F1=0.7375, Support=1341
I-ORG: Precision=0.9204, Recall=0.6312, F1=0.7488, Support=751
B-LOC: Precision=0.9205, Recall=0.8568, F1=0.8875, Support=1837
I-LOC: Precision=0.8959, Recall=0.7704, F1=0.8285, Support=257
B-MISC: Precision=0.8901, Recall=0.7993, F1=0.8423, Support=922
I-MISC: Precision=0.8780, Recall=0.6445, F1=0.7433, Support=346

Epoch 20/20, Loss: 0.002651331760518666
Training finished.

Evaluate after training...


100% 216/216 [00:00<00:00, 291.85it/s]


## Evaluation Results ##
Accuracy: 0.9341
Micro F1: 0.9341
Macro F1: 0.7473
## Detailed Performance by Entity Type ##
O: Precision=0.9607, Recall=0.9875, F1=0.9739, Support=38323
B-PER: Precision=0.8530, Recall=0.6314, F1=0.7257, Support=1617
I-PER: Precision=0.9179, Recall=0.5900, F1=0.7183, Support=1156
B-ORG: Precision=0.6038, Recall=0.7092, F1=0.6523, Support=1661
I-ORG: Precision=0.8564, Recall=0.6144, F1=0.7155, Support=835
B-LOC: Precision=0.8690, Recall=0.8076, F1=0.8372, Support=1668
I-LOC: Precision=0.8418, Recall=0.6420, F1=0.7285, Support=257
B-MISC: Precision=0.7919, Recall=0.7208, F1=0.7547, Support=702
I-MISC: Precision=0.7083, Recall=0.5509, F1=0.6198, Support=216

Evaluation finished.

Training and evaluation completed.


In [ ]:
results = evaluate(model, test_dataloader, device, train_dataloader.dataset.id2tag)

100% 216/216 [00:00<00:00, 290.75it/s]


## Evaluation Results ##
Accuracy: 0.9341
Micro F1: 0.9341
Macro F1: 0.7473
## Detailed Performance by Entity Type ##
O: Precision=0.9607, Recall=0.9875, F1=0.9739, Support=38323
B-PER: Precision=0.8530, Recall=0.6314, F1=0.7257, Support=1617
I-PER: Precision=0.9179, Recall=0.5900, F1=0.7183, Support=1156
B-ORG: Precision=0.6038, Recall=0.7092, F1=0.6523, Support=1661
I-ORG: Precision=0.8564, Recall=0.6144, F1=0.7155, Support=835
B-LOC: Precision=0.8690, Recall=0.8076, F1=0.8372, Support=1668
I-LOC: Precision=0.8418, Recall=0.6420, F1=0.7285, Support=257
B-MISC: Precision=0.7919, Recall=0.7208, F1=0.7547, Support=702
I-MISC: Precision=0.7083, Recall=0.5509, F1=0.6198, Support=216



In [ ]:
wandb.finish()

# Homework: Optimization, Activation

### Optimization 구현

1. Momentum, NesterovMomentum, AdaGrad, RMSProp, Adam, AdamW
2. AdamW는 강의자료에 없음. 인터넷 검색으로 찾아볼 것

### Activation 구현

1. sigmoid, tanh, ReLU

### 평가 기준

1. Optimization, Activation 당 5 - 45
2. 코드실행 - 10
3. 보고서 - 20
4. Hyperparameter tuning(max epoch: 20)
    25: 93점 이상
    20: 90점 이상
    10: 85점 이상
    0: 85점 미만

# Report: Optimization and Activation for NER with TextCNN

## 1. Introduction

### 연구 목적 및 배경

자연어 처리(NLP) 분야에서 개체명 인식(Named Entity Recognition, NER)은 문장에서 고유 명사(사람, 장소, 기관 등)를 식별하는 핵심 태스크 중 하나로,  
정보 추출, 검색 시스템, 대화형 인공지능 등의 기반 기술로 활용된다.

이번 과제에서는 **TextCNN 기반 NER 모델을 설계하고, 최적화 알고리즘과 활성화 함수들을 직접 구현하여 성능에 미치는 영향을 분석**하는 것을 목표로 한다.  
단순한 구현을 넘어서 각 구성 요소가 학습 효율성과 정확도에 어떤 기여를 하는지를 심층적으로 이해하는 데에 목적이 있다.


## 2. Methodology

### 2.1 데이터셋 개요

본 과제에서 사용된 데이터는 **ConLL 2003 NER 데이터셋**으로, 토큰화된 문장과 각 토큰의 개체명 태그를 포함한다.  
총 9개의 NER 클래스가 존재하며, BIO 스킴(Begin-Inside-Outside)을 따른다:

- `O`, `B-PER`, `I-PER`, `B-LOC`, `I-LOC`, `B-ORG`, `I-ORG`, `B-MISC`, `I-MISC`

### 2.2 모델 아키텍처: TextCNN 기반 NER

기본 모델은 **Kim (2014)의 TextCNN** 구조를 기반으로 하며, 단어 임베딩 후 1D Convolution 연산을 통해 특징을 추출한다.  
본 구현에서는 특징 추출을 이중적으로 수행하여 두 종류의 임베딩(Word2Vec과 Random)을 동시에 처리한다:

- **SG 임베딩**: Skip-Gram 방식의 Word2Vec으로 학습된 고정 임베딩
- **RD 임베딩**: 학습 가능한 랜덤 초기화 임베딩

각 임베딩에 대해 커널 크기 3, 5, 7의 Conv 레이어 3개씩 사용하여 총 6개 채널에서 특징을 추출한 뒤, 이를 concat하고 Linear 레이어를 통해 태그 분류를 수행한다.

### 2.3 Activation Functions

활성화 함수는 학습의 비선형성을 도입하여 복잡한 패턴을 모델이 학습할 수 있도록 돕는다. 직접 구현한 세 가지 함수는 다음과 같다:

- **ReLU (Rectified Linear Unit)**: $f(x) = max(0, x)$  
  ➤ sparsity 유도 및 gradient vanishing 방지  
- **Sigmoid**: $f(x) = \frac{1}{(1 + e^{-x})}$  
  ➤ 확률적 해석 가능하나 saturation 문제 있음  
- **Tanh**: $f(x) = tanh(x)$  
  ➤ 중심이 0이라 더 안정적이나 여전히 gradient가 소실됨

실험에서는 ReLU 사용 시 수렴 속도와 최종 성능에서 가장 우수한 결과를 보였다.

### 2.4 Optimizer 구현 및 설명

다음 6가지 최적화 알고리즘을 직접 구현하였다:

- **Momentum**: 관성을 부여하여 local minima 극복 가능  
- **Nesterov Momentum**: lookahead 기법으로 미리 가중치 이동  
- **AdaGrad**: 학습률을 feature마다 조정, 희소 데이터에 적합  
- **RMSProp**: 지수 이동 평균 기반 학습률 스케일링  
- **Adam**: Momentum + RMSProp을 결합한 강력한 optimizer  
- **AdamW**: Adam에 weight decay를 명확히 분리 적용 (L2 regularization과 구별)

이번 실험에서는 **AdamW**를 최종 선택하였으며, 과적합을 방지하는 효과와 안정적인 수렴을 보였다.

### 2.5 하이퍼파라미터 설정 근거

| Parameter        | Value     | Reason |
|------------------|-----------|--------|
| `batch_size`     | 16        | 일반적인 NER task에 적절, 작은 GPU 사용 고려 |
| `learning_rate`  | 5e-3      | Adam 계열에 적합한 비교적 큰 값, 빠른 수렴 유도 |
| `num_epochs`     | 20        | 과제 기준 최대, 학습 충분히 가능 |
| `max_len`        | 128       | 대부분 문장 길이 포괄 가능 |
| `embedding_dim`  | 32        | W2V 성능 대비 속도 고려 |
| `hidden_dim`     | 32        | Feature concatenation 시 충분한 채널 수 확보 |

---

## 3. Results & Analysis

### 3.1 성능 지표

학습 이후 모델은 다음과 같은 결과를 보였다:

- **최종 Test Accuracy**: `0.9341`
- **Micro F1 Score**: `0.9341`  
- **Macro F1 Score**: `0.7473`

### 3.2 클래스별 성능

| Class | F1 Score | Precision | Recall | Support |
|-------|----------|-----------|--------|---------|
| O     | 0.9739   | 0.9607    | 0.9875 | 38323   |
| B-PER | 0.7257   | 0.8530    | 0.6314 | 1617    |
| I-PER | 0.7183   | 0.9179    | 0.5900 | 1156    |
| B-ORG | 0.6523   | 0.6038    | 0.7092 | 1661    |
| I-ORG | 0.7155   | 0.8564    | 0.6144 | 835     |
| B-LOC | 0.8372   | 0.8690    | 0.8076 | 1668    |
| I-LOC | 0.7285   | 0.8418    | 0.6420 | 257     |
| B-MISC| 0.7547   | 0.7919    | 0.7208 | 702     |
| I-MISC| 0.6198   | 0.7083    | 0.5509 | 216     |

**해석**:
- `O`, `LOC`, `PER` 클래스의 성능이 우수하며, 주로 **데이터 분포가 큰 클래스**일수록 모델이 잘 학습함
- `I-MISC`는 데이터 수가 적고 변별력이 낮아 성능이 낮은 편

### 3.3 Training Loss 감소

- Epoch 1: 0.334 → Epoch 20: 0.0026  
→ 빠른 수렴과 안정적인 감소, 과적합 증거는 없음

---

## 4. Conclusion

### 4.1 성과 요약

- **Optimizer 6종, Activation 3종 직접 구현**하여 실제 모델에 적용
- TextCNN + Word2Vec + ReLU + AdamW 조합으로 **최고 93.4% 성능** 달성
- 직접 구현한 optimizer/activation이 PyTorch 내장과 동등한 성능을 보임을 확인

### 4.2 한계점 및 개선 방향

#### 한계점
- `ORG`, `MISC` 등의 클래스에서는 여전히 F1이 낮음 → 클래스 불균형
- 모든 파라미터 수동 설정으로 최적 조합 찾기 어려움

#### 개선 방향
- **Class balancing (oversampling / loss weighting)** 기법 적용
- **CRF layer** 추가로 예측 간의 관계 반영
- Word2Vec 외에 **contextual embedding (e.g., BERT)** 도입 고려
- CNN 구조 대신 **BiLSTM 또는 Transformer** 구조 확장 실험

---

이상으로 본 과제의 전체적인 구현과 실험을 마쳤으며, 직접 구현된 구성 요소들이 실제 모델 학습 및 성능에 긍정적인 영향을 주었음을 수치적으로 증명하였다.
